# Quantum Oracles

## An **oracle** is a black-box subroutine that recognises solutions to a problem. In quantum algorithms, oracles are implemented as unitaries that we *query in superposition*. They are the interface between a classical decision problem and a quantum algorithm.

# 1. Two equivalent oracle styles

## For a Boolean predicate $f : \{0,1\}^n \to \{0,1\}$, there are two standard unitary forms:

## - **Bit oracle** $O_f$: $\;|x\rangle|y\rangle \mapsto |x\rangle|y \oplus f(x)\rangle$. (We've used this in Deutsch–Jozsa.)
## - **Phase oracle** $O_f^{\pm}$: $\;|x\rangle \mapsto (-1)^{f(x)}|x\rangle$.

## Phase oracles are what Grover and most search-style algorithms use. Bit oracles are what most concrete circuit implementations build first.

# 2. Converting bit oracle → phase oracle

## Apply the bit oracle to an ancilla initialised in $|-\rangle$. By phase kickback (chapter 4 notebook 1):

$$ \Large  O_f \,(|x\rangle|{-}\rangle) = (-1)^{f(x)}|x\rangle |{-}\rangle. $$

## Discard the ancilla and what remains is the phase oracle acting on the data register. So **any bit oracle gives a phase oracle for free**, modulo one ancilla qubit.

In [ ]:
# Phase-kickback in numpy
import numpy as np

ket_minus = np.array([1, -1], dtype=complex) / np.sqrt(2)

def bit_oracle(f, n):
    """Bit oracle |x>|y> -> |x>|y XOR f(x)> as a 2^(n+1) x 2^(n+1) matrix."""
    U = np.zeros((2**(n+1), 2**(n+1)))
    for x in range(2**n):
        for y in (0, 1):
            in_idx  = (x << 1) | y
            out_y   = y ^ f(x)
            out_idx = (x << 1) | out_y
            U[out_idx, in_idx] = 1
    return U

# Test on f(x) = x_0 (first bit)
n = 2
f = lambda x: x & 1
U = bit_oracle(f, n)

# Prepare an arbitrary input state otimes |->
data = np.array([1, 2, 3, 4], dtype=complex); data /= np.linalg.norm(data)
joint = np.kron(data, ket_minus)
out = U @ joint
# Reshape and inspect the ancilla — it should still be |->
out = out.reshape(2**n, 2)
for x in range(2**n):
    print(f'x={x}: state on ancilla = {out[x]}    sign = {(-1)**f(x)}')

# 3. Building oracles for real predicates

## In practice you build a phase oracle by writing the predicate as a reversible circuit on classical bits, lifting it to a quantum circuit with Toffolis, then conjugating with $H \cdot X$ on the output ancilla. For example, the predicate *"$x$ encodes a satisfying assignment to a 3-SAT formula"* becomes a quantum circuit that:

## 1. evaluates each clause into ancilla qubits,
## 2. ANDs all clause-bits together using a multi-controlled Toffoli,
## 3. applies $Z$ on the resulting bit,
## 4. uncomputes step 1 (so ancillas return to $|0\rangle$).

## Step 4 — **uncomputation** — is essential, otherwise the oracle would entangle the answer with garbage and destroy interference.

# 4. Cost of an oracle

## When we say "Grover finds a marked item in $O(\sqrt{N})$ queries," we are counting *oracle calls*. Each call is however expensive your predicate is to implement — building the oracle dominates the real cost for most problems. A common pitfall is to assume the oracle is free, then conclude (wrongly) that an algorithm beats the best classical method.

# Recap

## - Bit oracle: $|x\rangle|y\rangle \to |x\rangle|y \oplus f(x)\rangle$.
## - Phase oracle: $|x\rangle \to (-1)^{f(x)}|x\rangle$. Get it from bit oracle + $|-\rangle$ ancilla.
## - Build oracles from reversible circuits + Toffolis + uncomputation.
## - Oracle cost matters: real speedups must include the oracle's gate count.

## Next: Grover's algorithm uses a phase oracle as its core.